# Reference
[uproot documentation](https://uproot.readthedocs.io/en/latest/)

In [ ]:
from uproot_io import Events, View
import numpy as np

In [ ]:
# Interaction type lookup
import csv
interaction_dictionary = {}
with open('interactions.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        key = int(row.pop('Idx'))
        interaction = row.pop('Interaction')
        interaction_dictionary[key] = interaction

In [ ]:
events = Events("TrainingFile.root")

In [ ]:
view_u = View(events, "u")
view_v = View(events, "v")
view_w = View(events, "w")

# File/Class contents - Events

What are the contents of the two classes?

The Events class is designed to provide information about PFOs, so each entry in the object is a single PFO from an event, which means all information from one event is spread across multiple rows. This means that event-level information, such as the number of final state particles in an event, is duplicated within each PFO in that event.

## Event-level and hierarchy information

`events.event_number` - Unique identifier for an **event**. Each event can (typically will) have multiple PFOs, so as noted above, this value will likely appear in several consecutive rows of the events object.

`events.interaction_type` - The underlying true interaction type of the event. A numeric value that can be converted to a text description via `interaction_dictionary` lookup table, which will have the form CCQEL_MU_P_P_P, or more generally whether the event is charged current (CC), neutral current (NC), the interaction type (e.g. QEL for quasielastic, DIS for deep inelastic scattering, etc), then a list of final state particles. For a CC interaction, the first of these if the leading lepton indicating if the incident neutrino was a muon or an electron (anti)neutrino, the remainder will be the other final state particles, so MU_P_P_P indicates a muon and 3 protons in the final state. You'll also come across OTHER_INTERACTION when the events get particularly complicated.

`events.num_final_state_particles` - The number of final state particles in the event

`events.reco_particle_index` - A unique identifier for each PFO in **within an event**

`events.reco_parent_index` - The reco particle index of this PFO's parent PFO (zero if the PFO is leading)

`events.reco_children_indices` - The an array of reco particle indices for this PFO's child PFOs (empty array if the PFO has no children)

## Hit information

`events.reco_num_hits_*` - The number of hits in a PFO separated by total, U, V and W views (these can be zero)

`events.reco_hits_x_*` - Arrays of drift coordinates for each hit in the PFO, separated by U, V and W views

`events.reco_hits_*` - Arrays of wire coordinates for each hit in the PFO, separated by U, V and W views (these coordinates will align with the corresponding reco_hits_x_* array)

`events.reco_adcs_*` - Arrays of charge deposition for each hit in the PFO, separated by U, V and W views (again, these coordinates will align with the corresponding reco_hits arrays, such that all information about a hit is conveyed by (drift coord, wire coord, adc)

`events.reco_hits_3d_*` - When different views in a PFO are successfully matched, the 3D location projecting into the three wire planes can be inferred. This variable contains arrays of those inferred 3D hit positions separated in x, y and z

## Vertex information

`events.reco_particle_vtx_3d_*` - The reconstructed 3D vertex location for each PFO separated by x, y and z coordinates

`events.reco_particle_vtx_*` - The reconstruced 3D vertex location above, projected into each view such that there are x, u, v and w values, x being the common drift coordinate shared by the views, and u, v and w being the correpsonding wire coordinate in the relevant view

## MC particle information

Each PFO should represent an underlying true particle, so information about the MC particle it was matched to is also included (not all PFOs successfully match to an MC particle)

`events.mc_num_hits_*` - The number of hits in the best matched MC particle separated by total, U, V and W views (if no match was found, this will be -1)

`events.mc_pdg` - The PDG code of the best matched MC particle (zewro if no match was found)


## Metrics

You may find it useful to consider the quality of the reconstructed particles you are working with, purity and completeness are defined for each PFO

`events.purity` - The fraction of hits in the PFO that actually match back to the best matched MC particle
`events.completeness` - The fraction of hits in the best matched MC particle that are included in the PFO

## Neutrino vertex

In addition to individual PFO vertices, we also include the reconstructed neutrino vertex

`events.neutrino_vtx_3d_*` - The reconstructed 3D vertex location for the incident neutrino separated by x, y and z coordinates

`events.neutrino_vtx_*` - The reconstruced 3D vertex location above, projected into each view such that there are x, u, v and w values

`events.true_neutrino_vtx_3d_*` - The true 3D vertex location for the incident neutrino separated by x, y and z coordinates

`events.true_neutrino_vtx_*` - The true 3D vertex location above, projected into each view such that there are x, u, v and w values

# File/Class contents - View

The View class repackages some of the information from the Events class for convenient of plotting events (for example). In particular, it aggregates some information from the various PFOs in one event such that each entry in a view object represents all of the hit information in that event. Each View object represents a specific LArTPC view. In particular

`view_*.all_events` - the event number as defined in Event, but without duplication

`view_*.true_vtx_x` - the true neutrino vertex drift coordinate in the relevant view

`view_*.true_vtx_z` - the true neutrino vertex wire coordinate in the relevant view

`view_*.x` - an array of drift coordinates for each hit in the event

`view_*.z` - an array of wire coordinates for each hit in the event

`view_*.adc` - an array of charge deposition for each hit in the event

x, z and adc arrays will align to form effective (x, z, adc) triplets

# Display an event

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
titlesize = 20
labelsize = 14

In [ ]:
 %matplotlib inline

Event displays are useful. We encourage you to make extensive use of them in helping you to understand what you're algorithms are doing, both from the perspective of thinking through what you think your algorithm should do, where it might encounter problems, and also as a debugging tool - if you're trying to reconstruct a vertex location, plot it on the relevant event display, as this will be much easier to interpret than the raw numbers.

The example below plots an event display for event 0 in the W wire plane. This example uses x, z and ADC variables in View jointly (for Event, keep in mind most variables are sets of arrays, not single blocks).

The range of values of x and z covered by any given event is highly variable and so by default you can get some quite skewed aspect ratios from Python. So to help you understand what the events really look like it can be helpful to enforce a true aspect ratio, this is what get_fig_ratio supplies, and once you pick a size for a figure in x, you can use this to enforce a suitable scale in y via the figsize parameter of plt.subplots. You don't __have__ to do this, and sometimes using the true aspect ratio can be unhelpful if the ratio of z and x extents becomes very large/small, but if you have to think about angles between tracks this approach can prevent a lot of confusion.

You'll often want to provide standard formatting for axes, so this is providing by format_axis (note this assumes a single subplot, you'll want to call this in a loop if you are formatting multiple subplots, passing in each individual subplot axis).

In the event display below you can see the hits from event 16 as a 2D histogram in various shades of blue, which represent the ADC contribution of each hit, this is achieved by using the view.adc variable to weight the hits at the corresponding coordinate (view.x and view.z). I've chosen a colour map that I find easy to interpret (i.e. a colour gradient that is easily interpreted from low to high values), but many options are available.

The true vertex is also added via a call to axes.scatter.

In [ ]:
def get_fig_ratio(xx, zz):
    x_range = np.max(xx) - np.min(xx)
    z_range = np.max(zz) - np.min(zz)
    return z_range / x_range if x_range else 1

def format_axis(ax):
    global titlesize, labelsize
    ax.tick_params(axis='x', labelsize=labelsize)
    ax.tick_params(axis='y', labelsize=labelsize)

In [ ]:
# The original solutions module is not part of this folder, so plot_event is defined here
def plot_event(view, event):
    xx, zz, adc = view.x[event], view.z[event], view.adc[event]
    x_size = 12
    fig, ax = plt.subplots(figsize=(x_size, x_size * get_fig_ratio(xx, zz)))
    ax.hist2d(xx, zz, bins=200, weights=adc, cmap="Blues", cmin=1e-6)
    ax.scatter(view.true_vtx_x[event], view.true_vtx_z[event], c="r", marker="x", s=100, label="True vertex")
    ax.set_xlabel("x (drift)", fontsize=labelsize)
    ax.set_ylabel("z (wire)", fontsize=labelsize)
    ax.set_title(f"Event {event}", fontsize=titlesize)
    ax.legend(fontsize=labelsize)
    format_axis(ax)
    plt.show()

In [ ]:
plot_event(view_w, 16)